# GPT-2 source-level textual unpredictability: Research Papers

This notebook reuses the existing PDF reading-order, subsection extraction, and complete-paragraph chunking pipeline. It scores each chunk with **GPT-2 only**, calculates bits per character (BPC), bins chunk BPC values into frequency histograms capped at 8–30 bins, and estimates the mean, variance, skewness, and excess kurtosis for each paper from the binned frequencies.

Chunking is subsection-local: complete paragraphs are accumulated until reaching at least 200 words; a subsection shorter than 200 words becomes one chunk; no paragraph is split; and an undersized terminal remainder is merged into the previous chunk in the same subsection. Exact duplicate PDFs are detected by SHA-256 and only one copy is used as a source.

Enable Internet in Kaggle to download GPT-2 if it is not already cached.

In [1]:
# Install dependencies used by the PDF parser and GPT-2 scorer.
%pip install -q pymupdf transformers accelerate scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 66.1 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import re
import math
import json
import shutil
import hashlib
import traceback
import zipfile
import collections

import fitz
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

DATASET_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/paper_gpt2_moments")
EXTRACT_ROOT = OUTPUT_ROOT / "extracted_papers"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 200
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STRIDE = 512
LOWERCASE_FOR_LM = False
HISTOGRAM_METHOD = "fd"
MIN_BINS = 8
MAX_BINS = 30

HEADER_FRACTION = 0.055
FOOTER_FRACTION = 0.045
END_SECTION_PATTERNS = [
    r"^references\b", r"^bibliography\b", r"^acknowledg(?:e)?ments?\b",
    r"^credit authorship contribution statement\b", r"^declaration of competing interest\b",
    r"^declaration of interests\b", r"^data availability\b", r"^funding\b",
    r"^funding statement\b", r"^supplementary (?:data|material)\b", r"^appendix\b",
    r"^appendix [A-Z]\b", r"^author contributions?\b", r"^conflict of interest\b",
]
COMMON_UNNUMBERED_HEADINGS = {
    "abstract", "introduction", "background", "theoretical background", "literature review",
    "method", "methods", "methodology", "materials and methods", "results", "findings",
    "discussion", "general discussion", "discussion and conclusion", "conclusion", "conclusions",
    "limitations", "strengths & limitations", "current study", "the present study", "research methodology",
    "theoretical framework", "conceptual background", "recommendations", "implications", "future research",
}

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_ROOT)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Minimum chunk size:", MIN_WORDS, "words")

Dataset: /kaggle/input
Output: /kaggle/working/paper_gpt2_moments
Model: gpt2
Device: cuda
Minimum chunk size: 200 words


## 1. Discover, deduplicate, and record the research-paper PDFs

In [3]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            block = stream.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def get_pdf_metadata(path):
    with fitz.open(str(path)) as doc:
        metadata = dict(doc.metadata or {})
        page_count = len(doc)
    return metadata, page_count

zip_files = sorted(DATASET_ROOT.rglob("*.zip"))
preferred_zips = [p for p in zip_files if "random_papers" in p.name.lower()]
if preferred_zips:
    zips_to_extract = preferred_zips
elif len(zip_files) == 1:
    zips_to_extract = zip_files
elif not zip_files:
    zips_to_extract = []
else:
    raise RuntimeError(
        "Multiple ZIP files were found under /kaggle/input and none is named random_papers. "
        "Keep only the research-paper ZIP in the Kaggle dataset, or name it random_papers.zip."
    )

for zip_path in zips_to_extract:
    extract_dir = EXTRACT_ROOT / zip_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    marker = extract_dir / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(extract_dir)
        marker.write_text("ok", encoding="utf-8")

pdf_candidates = set(DATASET_ROOT.rglob("*.pdf")) | set(EXTRACT_ROOT.rglob("*.pdf"))
all_pdf_paths = sorted((p.resolve() for p in pdf_candidates), key=lambda p: (p.name.lower(), str(p)))
if not all_pdf_paths:
    raise FileNotFoundError("No PDFs or ZIPs containing PDFs were found under /kaggle/input.")

manifest_rows = []
for path in all_pdf_paths:
    try:
        metadata, page_count = get_pdf_metadata(path)
        manifest_rows.append({
            "file": path.name,
            "path": str(path),
            "sha256": sha256_file(path),
            "pages": page_count,
            "title": metadata.get("title", "") or "",
            "author": metadata.get("author", "") or "",
            "subject": metadata.get("subject", "") or "",
            "read_status": "SUCCESS",
            "error": "",
        })
    except Exception as exc:
        manifest_rows.append({
            "file": path.name, "path": str(path), "sha256": "", "pages": np.nan,
            "title": "", "author": "", "subject": "", "read_status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(OUTPUT_ROOT / "paper_manifest_all_files.csv", index=False, encoding="utf-8-sig")

usable = manifest_df[manifest_df["read_status"].eq("SUCCESS") & manifest_df["sha256"].ne("")].copy()
duplicate_report = (
    usable.groupby("sha256")["file"].agg(list).reset_index(name="files")
)
duplicate_report["n_files"] = duplicate_report["files"].str.len()
duplicate_report = duplicate_report[duplicate_report["n_files"] > 1].copy()
duplicate_report.to_csv(OUTPUT_ROOT / "exact_duplicate_report.csv", index=False, encoding="utf-8-sig")

# One unique PDF = one source-level random variable; exact duplicate files are excluded.
used = usable.sort_values(["file", "path"]).drop_duplicates("sha256", keep="first").copy()
used = used.sort_values(["file", "path"]).reset_index(drop=True)
used["paper_index"] = np.arange(1, len(used) + 1)
used["paper_id"] = [
    f"paper_{i:02d}_" + re.sub(r"[^A-Za-z0-9]+", "_", Path(name).stem).strip("_")
    for i, name in zip(used["paper_index"], used["file"])
]
paper_manifest = used[[
    "paper_index", "paper_id", "file", "path", "sha256", "pages", "title", "author", "subject"
]].copy()
paper_manifest.to_csv(OUTPUT_ROOT / "paper_manifest_used.csv", index=False, encoding="utf-8-sig")

print("PDF paths discovered:", len(all_pdf_paths))
print("Unique readable paper PDFs:", len(paper_manifest))
if not duplicate_report.empty:
    print("Exact duplicates excluded:")
    display(duplicate_report)
display(paper_manifest[["paper_index", "file", "pages", "title", "author"]])

PDF paths discovered: 29
Unique readable paper PDFs: 29


,paper_index,file,pages,title,author
0,1,1-s2.0-S0022440525001013-main.pdf,24,Reporting guidelines for single-case experimen...,Thomas R. Kratochwill
1,2,1-s2.0-S0022440525001189-main.pdf,20,Evaluating a pragmatic universal social behavi...,Wendy M. Reinke
2,3,1-s2.0-S0140197121001330-main.pdf,10,Relationships between neighborhood collective ...,Ji Hyeon Lee
3,4,1-s2.0-S0140197121001366-main.pdf,12,Measuring advanced theory of mind: Do story-ba...,Marta Białecka-Pikul
4,5,1-s2.0-S0160738326001635-main.pdf,12,Academic travel: A subcultural theory approach,Paolo Mura
5,6,1-s2.0-S0160738326001684-main.pdf,6,Valid uses of stated intentions in tourism res...,Sara Dolnicar
6,7,1-s2.0-S0190740926000447-main.pdf,15,Healthcare access among youth with disabilitie...,Leah P. Cheatham
7,8,1-s2.0-S019074092600071X-main.pdf,13,Children’s experience of a new approach to fam...,Srijani Datta
8,9,1-s2.0-S0195666326003582-main.pdf,12,"Capability, opportunity, and motivation for he...",Sebastian Isbanner
9,10,1-s2.0-S0195666326003594-main.pdf,11,Can liking be instructed? Evaluations of famil...,Katrijn Houben


## 2. Extract paper sections and paragraphs

In [4]:
WORD_RE = re.compile(
    r"(?u)\b[\wÀ-ÖØ-öø-ÿ]+(?:['’\-][\wÀ-ÖØ-öø-ÿ]+)*\b"
)

NUMBERED_HEADING_RE = re.compile(
    r"^(?:\d+(?:\.\d+)*|[IVXLC]+)\.?\s+\S+"
)

CAPTION_RE = re.compile(
    r"^(?:fig(?:ure)?|table)\s*[\dIVX]+[\.:]?",
    re.I,
)

PAGE_NUMBER_RE = re.compile(
    r"^\d{1,4}$"
)

END_HEADING_RX = [
    re.compile(
        pattern,
        re.I,
    )
    for pattern in END_SECTION_PATTERNS
]

def normalize_text(text):
    text = text.replace(
        "\xad",
        "",
    )
    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

def count_words(text):
    return len(
        WORD_RE.findall(text)
    )

def block_to_info(block):
    if block.get("type") != 0:
        return None

    spans = [
        sp
        for line in block.get(
            "lines",
            [],
        )
        for sp in line.get(
            "spans",
            [],
        )
        if sp.get("text", "").strip()
    ]

    if not spans:
        return None

    text = normalize_text(
        "".join(
            sp["text"]
            for sp in spans
        )
    )

    if not text:
        return None

    return {
        "text": text,
        "x0": min(
            sp["bbox"][0]
            for sp in spans
        ),
        "y0": min(
            sp["bbox"][1]
            for sp in spans
        ),
        "x1": max(
            sp["bbox"][2]
            for sp in spans
        ),
        "y1": max(
            sp["bbox"][3]
            for sp in spans
        ),
        "font": max(
            float(
                sp.get(
                    "size",
                    0.0,
                )
            )
            for sp in spans
        ),
        "bold": any(
            (
                "bold"
                in sp.get(
                    "font",
                    "",
                ).lower()
                or
                "semibold"
                in sp.get(
                    "font",
                    "",
                ).lower()
            )
            for sp in spans
        ),
    }

def page_blocks_reading_order(page):
    infos = []

    for block in page.get_text(
        "dict"
    ).get(
        "blocks",
        [],
    ):
        info = block_to_info(
            block
        )

        if info is not None:
            infos.append(info)

    if not infos:
        return []

    page_width = page.rect.width
    page_height = page.rect.height

    infos = [
        b
        for b in infos
        if (
            b["y0"]
            >= HEADER_FRACTION * page_height
            and
            b["y1"]
            <= (
                1.0 - FOOTER_FRACTION
            ) * page_height
        )
    ]

    if not infos:
        return []

    left = []
    right = []
    full = []

    for b in infos:
        if (
            b["x0"]
            < 0.08 * page_width
            and
            b["x1"]
            > 0.92 * page_width
        ):
            full.append(b)

        elif (
            b["x0"] + b["x1"]
        ) / 2 < page_width / 2:
            left.append(b)

        else:
            right.append(b)

    if left and right:
        ordered_columns = (
            sorted(
                left,
                key=lambda x: (
                    x["y0"],
                    x["x0"],
                ),
            )
            +
            sorted(
                right,
                key=lambda x: (
                    x["y0"],
                    x["x0"],
                ),
            )
        )

        if full:
            first_column_y = min(
                b["y0"]
                for b in ordered_columns
            )

            if (
                min(
                    b["y0"]
                    for b in full
                )
                <= first_column_y
            ):
                return (
                    sorted(
                        full,
                        key=lambda x: x["y0"],
                    )
                    + ordered_columns
                )

            return sorted(
                ordered_columns + full,
                key=lambda x: x["y0"],
            )

        return ordered_columns

    return sorted(
        infos,
        key=lambda x: (
            x["y0"],
            x["x0"],
        ),
    )

def is_end_heading(text):
    return any(
        rx.match(text)
        for rx in END_HEADING_RX
    )

def is_toc_line(text):
    return bool(
        re.search(
            r"\.{4,}\s*\d+\s*$",
            text,
        )
    )

def is_caption(text):
    return bool(
        CAPTION_RE.match(text)
    )

def is_standalone_equation(text):
    if len(text) < 8:
        return False

    words = count_words(text)

    letters = sum(
        ch.isalpha()
        for ch in text
    )

    symbols = sum(
        not (
            ch.isalnum()
            or ch.isspace()
            or ch in ".,;:'’()-[]"
        )
        for ch in text
    )

    return (
        words < 12
        and
        letters / max(
            len(text),
            1,
        ) < 0.35
        and
        symbols / max(
            len(text),
            1,
        ) > 0.12
    )

def is_heading(block):
    text = block["text"]
    lower = text.lower().rstrip(":")

    if len(text) > 180:
        return False

    if is_caption(text):
        return False

    if NUMBERED_HEADING_RE.match(
        text
    ):
        return True

    if lower in COMMON_UNNUMBERED_HEADINGS:
        return True

    if (
        block["bold"]
        and
        block["font"] >= 7.5
        and
        not text.endswith(
            (
                ".",
                ",",
                ";",
                "?",
                "!",
            )
        )
    ):
        return True

    return False

def extract_paper_sections(pdf_path):
    doc = fitz.open(
        str(pdf_path)
    )

    page_blocks = []
    short_text_frequency = (
        collections.Counter()
    )
    contents_pages = set()

    for page_number, page in enumerate(
        doc,
        start=1,
    ):
        blocks = (
            page_blocks_reading_order(
                page
            )
        )

        for block in blocks:
            block["page"] = page_number

            if len(block["text"]) <= 120:
                short_text_frequency[
                    block["text"].lower()
                ] += 1

            if re.fullmatch(
                r"contents",
                block["text"],
                flags=re.I,
            ):
                contents_pages.add(
                    page_number
                )

        page_blocks.append(
            blocks
        )

    doc.close()

    repeated_short_text = {
        text
        for text, count
        in short_text_frequency.items()
        if count >= max(
            2,
            int(
                0.35
                * len(page_blocks)
            ),
        )
    }

    skip_until_page = (
        max(contents_pages) + 1
        if contents_pages
        else 0
    )

    sections = []
    current = None
    started = False

    for blocks in page_blocks:
        if not blocks:
            continue

        page_number = blocks[0][
            "page"
        ]

        if page_number < skip_until_page:
            continue

        for block in blocks:
            text = normalize_text(
                block["text"]
            )

            lower = text.lower()

            if not text:
                continue

            if PAGE_NUMBER_RE.fullmatch(
                text
            ):
                continue

            if is_toc_line(text):
                continue

            if (
                lower in repeated_short_text
                and len(text) < 120
            ):
                continue

            if is_end_heading(text):
                return sections

            if is_caption(text):
                continue

            if lower.startswith(
                "keywords"
            ):
                continue

            heading = is_heading(
                block
            )

            if heading:
                if not started:
                    if (
                        NUMBERED_HEADING_RE.match(
                            text
                        )
                        or
                        lower in {
                            "abstract",
                            "introduction",
                            "background",
                            "method",
                            "methods",
                            "methodology",
                            "materials and methods",
                            "text",
                        }
                        or
                        (
                            block["bold"]
                            and
                            block["font"] >= 7.5
                            and
                            block["page"] >= 2
                        )
                    ):
                        started = True
                    else:
                        continue

                current = {
                    "heading": text,
                    "page": block["page"],
                    "paragraphs": [],
                }

                sections.append(
                    current
                )

                continue

            if not started:
                continue

            if is_standalone_equation(
                text
            ):
                continue

            if (
                count_words(text) < 5
                and block["font"] < 8.5
            ):
                continue

            current["paragraphs"].append({
                "text": text,
                "page": block["page"],
                "font": block["font"],
                "bold": block["bold"],
            })

    return sections

## 3. Create complete-paragraph chunks within each subsection

In [5]:
def create_paper_chunks(
    sections,
    min_words=MIN_WORDS,
):
    """
    Chunk within each subsection only.

    Rules:
      1. A chunk contains complete paragraphs from one subsection.
      2. Accumulate paragraphs until >= min_words.
      3. A subsection with < min_words total becomes one chunk.
      4. A final undersized remainder is merged into the preceding
         chunk of the same subsection.
      5. No chunk crosses a subsection boundary.
      6. No paragraph is split.
    """
    chunks = []

    for section_index, section in enumerate(
        sections,
        start=1,
    ):
        paragraphs = [
            p
            for p in section["paragraphs"]
            if count_words(
                p["text"]
            ) > 0
        ]

        if not paragraphs:
            continue

        subsection_word_count = sum(
            count_words(
                p["text"]
            )
            for p in paragraphs
        )

        if subsection_word_count < min_words:
            chunks.append({
                "section_index": section_index,
                "section_heading": section["heading"],
                "paragraph_start": 1,
                "paragraph_end": len(
                    paragraphs
                ),
                "paragraph_count": len(
                    paragraphs
                ),
                "word_count": subsection_word_count,
                "start_page": paragraphs[0]["page"],
                "end_page": paragraphs[-1]["page"],
                "text": "\n\n".join(
                    p["text"]
                    for p in paragraphs
                ),
            })
            continue

        current = []
        current_words = 0
        subsection_chunks = []

        for paragraph_index, paragraph in enumerate(
            paragraphs,
            start=1,
        ):
            current.append(
                paragraph
            )

            current_words += count_words(
                paragraph["text"]
            )

            if current_words >= min_words:
                subsection_chunks.append({
                    "section_index": section_index,
                    "section_heading": section["heading"],
                    "paragraph_start": (
                        paragraph_index
                        - len(current)
                        + 1
                    ),
                    "paragraph_end": paragraph_index,
                    "paragraph_count": len(current),
                    "word_count": current_words,
                    "start_page": current[0]["page"],
                    "end_page": current[-1]["page"],
                    "text": "\n\n".join(
                        p["text"]
                        for p in current
                    ),
                })

                current = []
                current_words = 0

        if current:
            remainder_text = "\n\n".join(
                p["text"]
                for p in current
            )

            if subsection_chunks:
                previous = (
                    subsection_chunks[-1]
                )

                previous[
                    "paragraph_end"
                ] = len(
                    paragraphs
                )

                previous[
                    "paragraph_count"
                ] += len(current)

                previous[
                    "word_count"
                ] += current_words

                previous[
                    "end_page"
                ] = current[-1]["page"]

                previous["text"] += (
                    "\n\n"
                    + remainder_text
                )

            else:
                subsection_chunks.append({
                    "section_index": section_index,
                    "section_heading": section["heading"],
                    "paragraph_start": 1,
                    "paragraph_end": len(
                        paragraphs
                    ),
                    "paragraph_count": len(
                        paragraphs
                    ),
                    "word_count": current_words,
                    "start_page": current[0]["page"],
                    "end_page": current[-1]["page"],
                    "text": remainder_text,
                })

        chunks.extend(
            subsection_chunks
        )

    for chunk_id, chunk in enumerate(
        chunks,
        start=1,
    ):
        chunk["chunk_id"] = chunk_id

    return chunks

## 4. Load GPT-2 and define next-token scoring

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

possible_lengths = [
    getattr(tokenizer, "model_max_length", None),
    getattr(model.config, "n_positions", None),
    getattr(model.config, "max_position_embeddings", None),
]
valid_lengths = [x for x in possible_lengths if isinstance(x, int) and 0 < x < 100000]
MODEL_CONTEXT = min(valid_lengths) if valid_lengths else 1024

print("Loaded:", MODEL_NAME)
print("Context length:", MODEL_CONTEXT)
print("Device:", DEVICE)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded: gpt2
Context length: 1024
Device: cuda


In [7]:
def prepare_lm_text(text):
    return (
        text.lower()
        if LOWERCASE_FOR_LM
        else text
    )

def score_text_lm(text):
    text_for_model = prepare_lm_text(
        text
    )

    encoded = tokenizer(
        text_for_model,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded[
        "input_ids"
    ]

    sequence_length = (
        input_ids.size(1)
    )

    if sequence_length < 2:
        return {
            "total_bits": np.nan,
            "scored_tokens": 0,
            "bits_per_token": np.nan,
            "bpc": np.nan,
            "perplexity": np.nan,
            "model_tokens": sequence_length,
            "character_count": len(
                text_for_model
            ),
        }

    total_nll_nats = 0.0
    scored_tokens = 0

    previous_end = 1

    with torch.no_grad():
        for begin in range(
            0,
            sequence_length - 1,
            STRIDE,
        ):
            end = min(
                begin + MODEL_CONTEXT,
                sequence_length,
            )

            window = input_ids[
                :,
                begin:end,
            ].to(DEVICE)

            outputs = model(
                input_ids=window
            )

            logits = outputs.logits

            shift_logits = logits[
                :,
                :-1,
                :,
            ]

            shift_labels = window[
                :,
                1:,
            ]

            absolute_target_start = max(
                previous_end,
                begin + 1,
            )

            relative_target_start = (
                absolute_target_start
                - begin
            )

            if (
                relative_target_start
                > shift_labels.size(1)
            ):
                continue

            shifted_start = (
                relative_target_start - 1
            )

            selected_logits = (
                shift_logits[
                    :,
                    shifted_start:,
                    :,
                ]
            )

            selected_labels = (
                shift_labels[
                    :,
                    shifted_start:,
                ]
            )

            log_probs = torch.log_softmax(
                selected_logits,
                dim=-1,
            )

            target_log_probs = (
                log_probs.gather(
                    2,
                    selected_labels.unsqueeze(
                        -1
                    ),
                ).squeeze(-1)
            )

            total_nll_nats += float(
                -target_log_probs.sum().item()
            )

            scored_here = int(
                selected_labels.numel()
            )

            scored_tokens += (
                scored_here
            )

            previous_end = end

            if end >= sequence_length:
                break

    total_bits = (
        total_nll_nats
        / math.log(2)
    )

    bits_per_token = (
        total_bits / scored_tokens
        if scored_tokens
        else np.nan
    )

    mean_nll_nats = (
        total_nll_nats
        / scored_tokens
        if scored_tokens
        else np.nan
    )

    perplexity = (
        math.exp(mean_nll_nats)
        if (
            np.isfinite(
                mean_nll_nats
            )
            and mean_nll_nats < 50
        )
        else np.inf
    )

    chars = len(
        text_for_model
    )

    bpc = (
        total_bits / chars
        if chars
        else np.nan
    )

    return {
        "total_bits": total_bits,
        "scored_tokens": scored_tokens,
        "bits_per_token": bits_per_token,
        "bpc": bpc,
        "perplexity": perplexity,
        "model_tokens": sequence_length,
        "character_count": chars,
    }

## 5. Bin chunk BPC frequencies and calculate source-level moments

In [8]:
def frequency_histogram(values, max_bins=MAX_BINS, min_bins=MIN_BINS):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("No finite BPC values to bin.")

    if values.size == 1 or np.isclose(values.min(), values.max()):
        center = float(values[0])
        edges = np.array([center - 0.5, center + 0.5], dtype=float)
    else:
        fd_edges = np.histogram_bin_edges(values, bins="fd")
        proposed_bins = max(1, len(fd_edges) - 1)
        n_bins = int(np.clip(proposed_bins, min_bins, max_bins))
        edges = np.linspace(float(values.min()), float(values.max()), n_bins + 1)

    counts, edges = np.histogram(values, bins=edges)
    centers = (edges[:-1] + edges[1:]) / 2
    keep = counts > 0
    return pd.DataFrame({
        "bin_left": edges[:-1][keep],
        "bin_right": edges[1:][keep],
        "bin_center": centers[keep],
        "frequency": counts[keep].astype(int),
    })


def moments_from_frequencies(hist):
    centers = hist["bin_center"].to_numpy(dtype=float)
    frequencies = hist["frequency"].to_numpy(dtype=float)
    n = float(frequencies.sum())
    if n <= 0:
        raise ValueError("Histogram has no observations.")

    mean = float(np.sum(frequencies * centers) / n)
    centered = centers - mean
    m2 = float(np.sum(frequencies * centered**2) / n)
    m3 = float(np.sum(frequencies * centered**3) / n)
    m4 = float(np.sum(frequencies * centered**4) / n)

    if m2 > 0:
        skewness = m3 / (m2 ** 1.5)
        excess_kurtosis = m4 / (m2 ** 2) - 3.0
    else:
        skewness = np.nan
        excess_kurtosis = np.nan

    return {
        "n_valid_chunks": int(n),
        "n_histogram_bins": int(len(hist)),
        "mean_bpc": mean,
        "variance_bpc": m2,
        "skewness_bpc": float(skewness) if np.isfinite(skewness) else np.nan,
        "excess_kurtosis_bpc": float(excess_kurtosis) if np.isfinite(excess_kurtosis) else np.nan,
    }

In [9]:
chunk_rows = []
frequency_rows = []
moment_rows = []

for paper in paper_manifest.itertuples(index=False):
    paper_index = int(paper.paper_index)
    pdf_path = Path(paper.path)
    paper_id = paper.paper_id
    print(f"\n[{paper_index:02d}/{len(paper_manifest):02d}] {paper.file}")

    try:
        sections = extract_paper_sections(pdf_path)
        if not sections:
            raise RuntimeError("No paper sections could be reconstructed.")
        chunks = create_paper_chunks(sections, MIN_WORDS)
        if not chunks:
            raise RuntimeError("No chunks were created.")

        values = []
        paper_chunk_rows = []
        for chunk in chunks:
            score = score_text_lm(chunk["text"])
            row = {
                "paper_index": paper_index,
                "paper_id": paper_id,
                "source_name": paper.title if str(paper.title).strip() else Path(paper.file).stem,
                "paper_file": paper.file,
                "paper_title": paper.title,
                "paper_author": paper.author,
                "source_pdf": str(pdf_path),
                "chunk_id": chunk["chunk_id"],
                "section_index": chunk["section_index"],
                "section_heading": chunk["section_heading"],
                "paragraph_start": chunk["paragraph_start"],
                "paragraph_end": chunk["paragraph_end"],
                "paragraph_count": chunk["paragraph_count"],
                "start_page": chunk["start_page"],
                "end_page": chunk["end_page"],
                "word_count": chunk["word_count"],
                "model_name": MODEL_NAME,
                "bpc": score["bpc"],
                "bits_per_token": score["bits_per_token"],
                "scored_tokens": score["scored_tokens"],
                "character_count": score["character_count"],
            }
            paper_chunk_rows.append(row)
            if np.isfinite(score["bpc"]):
                values.append(float(score["bpc"]))

        chunk_rows.extend(paper_chunk_rows)
        if not values:
            raise RuntimeError("No finite BPC values produced.")

        hist = frequency_histogram(values)
        hist.insert(0, "paper_id", paper_id)
        hist.insert(1, "source_name", paper.title if str(paper.title).strip() else Path(paper.file).stem)
        hist.insert(2, "model_name", MODEL_NAME)
        frequency_rows.extend(hist.to_dict("records"))
        stats = moments_from_frequencies(hist)

        moment_rows.append({
            "paper_index": paper_index,
            "paper_id": paper_id,
            "source_name": paper.title if str(paper.title).strip() else Path(paper.file).stem,
            "paper_file": paper.file,
            "paper_title": paper.title,
            "paper_author": paper.author,
            "source_pdf": str(pdf_path),
            "model_name": MODEL_NAME,
            "status": "SUCCESS",
            "pages": int(paper.pages),
            "n_sections": len(sections),
            "n_chunks": len(chunks),
            **stats,
        })
        print(f"  sections={len(sections)}, chunks={len(chunks)}, mean BPC≈{stats['mean_bpc']:.5f}")

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {exc}"
        print("  [ERROR]", error_text)
        moment_rows.append({
            "paper_index": paper_index,
            "paper_id": paper_id,
            "source_name": paper.title if str(paper.title).strip() else Path(paper.file).stem,
            "paper_file": paper.file,
            "paper_title": paper.title,
            "paper_author": paper.author,
            "source_pdf": str(pdf_path),
            "model_name": MODEL_NAME,
            "status": "ERROR",
            "pages": int(paper.pages) if pd.notna(paper.pages) else np.nan,
            "n_sections": np.nan,
            "n_chunks": np.nan,
            "n_valid_chunks": 0,
            "n_histogram_bins": 0,
            "mean_bpc": np.nan,
            "variance_bpc": np.nan,
            "skewness_bpc": np.nan,
            "excess_kurtosis_bpc": np.nan,
            "error": error_text,
        })

paper_moments_df = pd.DataFrame(moment_rows).sort_values("paper_index").reset_index(drop=True)
paper_chunks_df = pd.DataFrame(chunk_rows)
paper_frequencies_df = pd.DataFrame(frequency_rows)

moments_path = OUTPUT_ROOT / "paper_source_moments_gpt2.csv"
chunks_path = OUTPUT_ROOT / "paper_chunk_scores_gpt2.csv"
frequencies_path = OUTPUT_ROOT / "paper_binned_frequencies_gpt2.csv"

paper_moments_df.to_csv(moments_path, index=False, encoding="utf-8-sig")
paper_chunks_df.to_csv(chunks_path, index=False, encoding="utf-8-sig")
paper_frequencies_df.to_csv(frequencies_path, index=False, encoding="utf-8-sig")

print("\nMain output:", moments_path)
display(paper_moments_df)


[01/29] 1-s2.0-S0022440525001013-main.pdf
  sections=14, chunks=53, mean BPC≈1.03947

[02/29] 1-s2.0-S0022440525001189-main.pdf


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1152 > 1024). Running this sequence through the model will result in indexing errors


  sections=15, chunks=50, mean BPC≈1.01748

[03/29] 1-s2.0-S0140197121001330-main.pdf
  sections=6, chunks=19, mean BPC≈0.88725

[04/29] 1-s2.0-S0140197121001366-main.pdf
  sections=7, chunks=31, mean BPC≈1.09890

[05/29] 1-s2.0-S0160738326001635-main.pdf
  sections=11, chunks=32, mean BPC≈1.01189

[06/29] 1-s2.0-S0160738326001684-main.pdf
  sections=7, chunks=11, mean BPC≈1.23535

[07/29] 1-s2.0-S0190740926000447-main.pdf
  sections=7, chunks=51, mean BPC≈0.97261

[08/29] 1-s2.0-S019074092600071X-main.pdf
  sections=5, chunks=40, mean BPC≈1.04752

[09/29] 1-s2.0-S0195666326003582-main.pdf
  sections=9, chunks=25, mean BPC≈1.03487

[10/29] 1-s2.0-S0195666326003594-main.pdf
  sections=29, chunks=37, mean BPC≈1.00553

[11/29] 1-s2.0-S026151772600124X-main.pdf
  sections=10, chunks=12, mean BPC≈2.23578

[12/29] 1-s2.0-S0261517726001305-main.pdf
  sections=7, chunks=48, mean BPC≈1.20830

[13/29] 1-s2.0-S0277953626009718-main.pdf
  sections=6, chunks=18, mean BPC≈1.01969

[14/29] 1-s2.0-S02

,paper_index,paper_id,source_name,paper_file,paper_title,paper_author,source_pdf,model_name,status,pages,n_sections,n_chunks,n_valid_chunks,n_histogram_bins,mean_bpc,variance_bpc,skewness_bpc,excess_kurtosis_bpc
0,1,paper_01_1_s2_0_S0022440525001013_main,Reporting guidelines for single-case experimen...,1-s2.0-S0022440525001013-main.pdf,Reporting guidelines for single-case experimen...,Thomas R. Kratochwill,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,24,14,53,53,7,1.039466,0.009576,0.588549,-0.045610
1,2,paper_02_1_s2_0_S0022440525001189_main,Evaluating a pragmatic universal social behavi...,1-s2.0-S0022440525001189-main.pdf,Evaluating a pragmatic universal social behavi...,Wendy M. Reinke,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,20,15,50,50,8,1.017475,0.030094,1.831633,3.039437
2,3,paper_03_1_s2_0_S0140197121001330_main,Relationships between neighborhood collective ...,1-s2.0-S0140197121001330-main.pdf,Relationships between neighborhood collective ...,Ji Hyeon Lee,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,10,6,19,19,5,0.887253,0.027225,2.353129,6.092361
3,4,paper_04_1_s2_0_S0140197121001366_main,Measuring advanced theory of mind: Do story-ba...,1-s2.0-S0140197121001366-main.pdf,Measuring advanced theory of mind: Do story-ba...,Marta Białecka-Pikul,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,12,7,31,31,8,1.098898,0.023051,0.719878,-0.225271
4,5,paper_05_1_s2_0_S0160738326001635_main,Academic travel: A subcultural theory approach,1-s2.0-S0160738326001635-main.pdf,Academic travel: A subcultural theory approach,Paolo Mura,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,12,11,32,32,7,1.011893,0.006275,0.412768,0.209956
5,6,paper_06_1_s2_0_S0160738326001684_main,Valid uses of stated intentions in tourism res...,1-s2.0-S0160738326001684-main.pdf,Valid uses of stated intentions in tourism res...,Sara Dolnicar,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,6,7,11,11,5,1.235353,0.004048,-0.466627,-1.068534
6,7,paper_07_1_s2_0_S0190740926000447_main,Healthcare access among youth with disabilitie...,1-s2.0-S0190740926000447-main.pdf,Healthcare access among youth with disabilitie...,Leah P. Cheatham,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,15,7,51,51,8,0.972612,0.011129,0.147308,-0.936685
7,8,paper_08_1_s2_0_S019074092600071X_main,Children’s experience of a new approach to fam...,1-s2.0-S019074092600071X-main.pdf,Children’s experience of a new approach to fam...,Srijani Datta,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,13,5,40,40,8,1.047520,0.014858,0.580162,-0.225941
8,9,paper_09_1_s2_0_S0195666326003582_main,"Capability, opportunity, and motivation for he...",1-s2.0-S0195666326003582-main.pdf,"Capability, opportunity, and motivation for he...",Sebastian Isbanner,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,12,9,25,25,7,1.034870,0.028834,1.222687,0.663511
9,10,paper_10_1_s2_0_S0195666326003594_main,Can liking be instructed? Evaluations of famil...,1-s2.0-S0195666326003594-main.pdf,Can liking be instructed? Evaluations of famil...,Katrijn Houben,/kaggle/input/datasets/srabongitikar/some-rese...,gpt2,SUCCESS,11,29,37,37,8,1.005527,0.048975,1.715005,2.467956


## Output files

The main file is `paper_source_moments_gpt2.csv`. It contains one row per unique readable paper and the source-level `mean_bpc`, `variance_bpc`, `skewness_bpc`, and `excess_kurtosis_bpc`. A paper that fails during parsing or scoring remains in the output with its status and missing moment values. Chunk scores and binned frequencies are also exported for auditability.

In Kaggle, find the files under `/kaggle/working/paper_gpt2_moments/` and use **Save Version** with output saving enabled to download them.

In [10]:
import numpy as np
import pandas as pd
import math
import shutil
import subprocess
from pathlib import Path

plot_df = paper_moments_df.copy()

plot_df["n_valid_chunks"] = pd.to_numeric(
    plot_df["n_valid_chunks"], errors="coerce"
)
plot_df["mean_bpc"] = pd.to_numeric(
    plot_df["mean_bpc"], errors="coerce"
)
plot_df["variance_bpc"] = pd.to_numeric(
    plot_df["variance_bpc"], errors="coerce"
)

plot_df = plot_df[
    (plot_df["status"] == "SUCCESS")
    & (plot_df["n_valid_chunks"] > 1)
    & np.isfinite(plot_df["mean_bpc"])
    & np.isfinite(plot_df["variance_bpc"])
    & (plot_df["variance_bpc"] > 0)
].copy()

if len(plot_df) < 2:
    raise ValueError("Insufficient valid papers to plot.")

x = plot_df["mean_bpc"].to_numpy(dtype=float)
y = plot_df["variance_bpc"].to_numpy(dtype=float)

x_min, x_max = x.min(), x.max()
x_margin = 0.06 * (x_max - x_min)
if x_margin == 0:
    x_margin = max(abs(x_min) * 0.05, 0.05)

y_min = 10.0 ** (math.floor(math.log10(y.min())) - 1)
y_max = 10.0 ** math.ceil(math.log10(y.max()))

y_ticks = ",".join(
    str(k) for k in range(
        int(math.log10(y_min)),
        int(math.log10(y_max)) + 1
    )
)

coordinates = "\n".join(
    f"        ({xi:.8f},{yi:.10g})"
    for xi, yi in zip(x, y)
)

tex = rf"""
% Requires \usepackage{{pgfplots}} and \pgfplotsset{{compat=1.17}}

\begin{{tikzpicture}}
\begin{{axis}}[
    width=7.2cm,
    height=7.2cm,
    scale only axis,
    axis lines=box,
    axis line style={{black, line width=0.55pt}},
    tick align=outside,
    tick style={{black, line width=0.45pt}},
    xlabel={{Mean BPC (bits per character)}},
    ylabel={{Variance of BPC ($\mathrm{{BPC}}^2$)}},
    xmin={x_min - x_margin:.8f},
    xmax={x_max + x_margin:.8f},
    ymode=log,
    ymin={y_min:.10g},
    ymax={y_max:.10g},
    log basis y=10,
    log ticks with fixed point,
    ytickten={{{y_ticks}}},
    scaled y ticks=false,
    grid=major,
    major grid style={{gray!30, densely dotted, line width=0.35pt}},
    minor grid style={{gray!12, dotted, line width=0.25pt}},
    minor y tick num=9,
    tick label style={{font=\small}},
    label style={{font=\small}},
    clip=true,
]
\addplot[
    only marks,
    mark=*,
    mark size=1.7pt,
    mark options={{draw=black, fill=black}}
] coordinates {{
{coordinates}
}};
\end{{axis}}
\end{{tikzpicture}}
"""

output_dir = Path(OUTPUT_ROOT)
output_dir.mkdir(parents=True, exist_ok=True)

plot_tex = output_dir / "paper_mean_variance_pgfplots.tex"
plot_tex.write_text(tex.strip() + "\n", encoding="utf-8")

standalone_tex = output_dir / "paper_mean_variance_standalone.tex"
standalone_tex.write_text(
    r"""\documentclass[tikz,border=2pt]{standalone}
\usepackage{pgfplots}
\pgfplotsset{compat=1.17}
\begin{document}
\input{paper_mean_variance_pgfplots.tex}
\end{document}
""",
    encoding="utf-8",
)

print(f"Papers plotted: {len(plot_df)}")
print(f"PGFPlots source: {plot_tex}")
print(f"Standalone LaTeX source: {standalone_tex}")

if shutil.which("pdflatex"):
    result = subprocess.run(
        [
            "pdflatex",
            "-interaction=nonstopmode",
            "-halt-on-error",
            standalone_tex.name,
        ],
        cwd=output_dir,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print(f"Compiled vector PDF: {output_dir / 'paper_mean_variance_standalone.pdf'}")
    else:
        print("LaTeX compilation failed. The PGFPlots source was still saved.")
        print(result.stdout[-2500:])
else:
    print("pdflatex is unavailable; compile the standalone .tex file in your LaTeX environment.")

Papers plotted: 29
PGFPlots source: /kaggle/working/paper_gpt2_moments/paper_mean_variance_pgfplots.tex
Standalone LaTeX source: /kaggle/working/paper_gpt2_moments/paper_mean_variance_standalone.tex
pdflatex is unavailable; compile the standalone .tex file in your LaTeX environment.
